# CAD Code Generator — Colab driver

This notebook runs the refactored package on a Colab GPU. It assumes you have:
1. Pushed `cad-code-generator/` to GitHub.
2. Mounted Google Drive for checkpoint persistence (optional but recommended).

**Runtime**: GPU (T4 is fine; A100 if available).

## 1. Setup

In [ ]:
# Mount Drive so checkpoints survive runtime disconnects
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo
!git clone https://github.com/<your-user>/cad-code-generator.git
%cd cad-code-generator

In [ ]:
# CadQuery needs libgl
!sudo apt-get -qq install -y libgl1

# Install the package + deps (~3-5 min)
!pip install -q -e .

In [ ]:
# Sanity check
import torch, cadquery as cq
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('CadQuery:', cq.__version__)

## 2. Train the tokenizer (one-time, ~5 min)

In [ ]:
!python scripts/train_tokenizer.py --save-dir cadquery_tokenizer

## 3. Train the model

Set `--checkpoint-dir` to a Drive path so progress survives disconnects.

In [ ]:
!python scripts/train.py \
    --tokenizer-dir cadquery_tokenizer \
    --checkpoint-dir /content/drive/MyDrive/cad-codegen-checkpoints \
    --epochs 15 \
    --batch-size 128

## 4. Evaluate on the test set

In [ ]:
!python scripts/evaluate.py \
    --checkpoint /content/drive/MyDrive/cad-codegen-checkpoints/best.pt \
    --tokenizer-dir cadquery_tokenizer \
    --method greedy \
    --n-samples 200

## 5. Single-image inference

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from cad_codegen.inference import load_inference_pipeline, predict_code
from datasets import load_dataset

# Load pipeline
model, tokenizer, transform, device = load_inference_pipeline(
    checkpoint_path='/content/drive/MyDrive/cad-codegen-checkpoints/best.pt',
    tokenizer_dir='cadquery_tokenizer',
)

# Grab a sample from the test set
ds = load_dataset('CADCODER/GenCAD-Code', split='test', num_proc=4)
sample = ds[0]
img = sample['image']  # PIL.Image
gt_code = sample['cadquery']

# Predict
pred = predict_code(img, model, tokenizer, transform, device, method='beam')

plt.imshow(img); plt.axis('off'); plt.title(f"id: {sample['deepcad_id']}"); plt.show()
print('── GT ──'); print(gt_code)
print('── PRED ──'); print(pred)